# Cross-Student Comparison — Scale Worm Detection

## Purpose

This notebook compares results across all student analysts to assess:
1. **Inter-annotator agreement** — do students agree on what is/isn't a scale worm?
2. **Labeling effort** — how many corrections did each student make?
3. **Model convergence** — how quickly did each student's model stabilize?
4. **Final count agreement** — do the improved models agree on population counts?

**Run this after all students have completed their Phase 4 exports.**


In [ ]:
# ── Imports ──
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11


## Configuration

Edit the student list below. Each entry maps a student name to their assigned month.


In [ ]:
# ╔══════════════════════════════════════════════╗
# ║   EDIT: Student names and assigned months    ║
# ╚══════════════════════════════════════════════╝
STUDENTS = {
    "student_a": "2024/09",
    "student_b": "2024/10",
    "student_c": "2024/11",
    "student_d": "2024/12",
}

STUDENT_ROOT = Path.home() / "scaleworm_students"


---
## 1. Load Student Results

Scan each student's output directory for report cards, corrections, and counts.


In [ ]:
# ── Discover student outputs ──
student_data = {}

for name, month in STUDENTS.items():
    sdir = STUDENT_ROOT / name
    if not sdir.exists():
        print(f"WARNING: {name} directory not found at {sdir}")
        continue

    info = {"name": name, "month": month, "rounds": []}

    # Find all round directories
    month_dir = sdir / month.replace("/", "_")
    if not month_dir.exists():
        # Try alternative layout
        month_dir = sdir
    round_dirs = sorted(month_dir.glob("round_*"))
    if not round_dirs:
        round_dirs = sorted(sdir.glob("*/round_*"))

    for rd in round_dirs:
        round_info = {"dir": rd, "round": int(rd.name.split("_")[1])}

        # Corrections
        corr_path = rd / "corrections.json"
        if corr_path.exists():
            with open(corr_path) as f:
                corr = json.load(f)
            round_info["corrections"] = corr
            round_info["n_fps"] = sum(len(v.get("false_positives", [])) for v in corr.values())
            round_info["n_fns"] = sum(len(v.get("missed_worms", [])) for v in corr.values())
            round_info["n_frames_reviewed"] = len(corr)
        else:
            round_info["corrections"] = {}
            round_info["n_fps"] = 0
            round_info["n_fns"] = 0
            round_info["n_frames_reviewed"] = 0

        # Counts
        counts_path = rd / "counts.parquet"
        if counts_path.exists():
            round_info["counts"] = pd.read_parquet(counts_path)
        elif (rd / "counts_new.parquet").exists():
            round_info["counts"] = pd.read_parquet(rd / "counts_new.parquet")

        info["rounds"].append(round_info)

    # Final export
    final_dir = sdir / "final"
    if final_dir.exists():
        info["has_final"] = True
        final_counts = final_dir / "final_counts.parquet"
        if final_counts.exists():
            info["final_counts"] = pd.read_parquet(final_counts)
        report_card = final_dir / "report_card.json"
        if report_card.exists():
            with open(report_card) as f:
                info["report_card"] = json.load(f)
    else:
        info["has_final"] = False

    student_data[name] = info
    print(f"  {name}: {len(info['rounds'])} rounds, final={'YES' if info['has_final'] else 'NO'}")

print(f"\nLoaded data for {len(student_data)} students.")


---
## 2. Labeling Effort

How many corrections did each student make per round? More corrections generally
indicate more careful annotation, though it also depends on model quality.


In [ ]:
# ── Labeling effort summary ──
effort_rows = []
for name, info in student_data.items():
    for ri in info["rounds"]:
        effort_rows.append({
            "student": name,
            "round": ri["round"],
            "frames_reviewed": ri["n_frames_reviewed"],
            "false_positives": ri["n_fps"],
            "missed_worms": ri["n_fns"],
            "total_corrections": ri["n_fps"] + ri["n_fns"],
        })

effort_df = pd.DataFrame(effort_rows)
if len(effort_df):
    print(effort_df.to_string(index=False))
else:
    print("No correction data found yet.")


In [ ]:
# ── Labeling effort figure ──
if len(effort_df):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    students = list(student_data.keys())
    colors = plt.cm.tab10(np.arange(len(students)))
    student_colors = dict(zip(students, colors))

    # Total corrections per round
    ax = axes[0]
    for name in students:
        sub = effort_df[effort_df["student"] == name]
        ax.plot(sub["round"], sub["total_corrections"], "o-",
                color=student_colors[name], label=name)
    ax.set_xlabel("Round")
    ax.set_ylabel("Total corrections")
    ax.set_title("Corrections per Round")
    ax.legend()

    # FP vs FN breakdown
    ax = axes[1]
    x = np.arange(len(students))
    fp_totals = [effort_df[effort_df["student"] == s]["false_positives"].sum() for s in students]
    fn_totals = [effort_df[effort_df["student"] == s]["missed_worms"].sum() for s in students]
    ax.bar(x - 0.15, fp_totals, 0.3, label="False positives", color="salmon")
    ax.bar(x + 0.15, fn_totals, 0.3, label="Missed worms", color="steelblue")
    ax.set_xticks(x)
    ax.set_xticklabels(students, rotation=30, ha="right")
    ax.set_ylabel("Total corrections")
    ax.set_title("FP vs FN by Student")
    ax.legend()

    # Frames reviewed
    ax = axes[2]
    frames_total = [effort_df[effort_df["student"] == s]["frames_reviewed"].sum() for s in students]
    ax.bar(x, frames_total, color=[student_colors[s] for s in students])
    ax.set_xticks(x)
    ax.set_xticklabels(students, rotation=30, ha="right")
    ax.set_ylabel("Total frames reviewed")
    ax.set_title("Review Effort")

    plt.tight_layout()
    plt.show()


---
## 3. Model Convergence

Track how each student's model improves across rounds. Convergence means the
corrections made in each round decrease — the model is learning from student feedback.


In [ ]:
# ── Convergence trajectories ──
if len(effort_df):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Correction rate (corrections per frame)
    ax = axes[0]
    for name in students:
        sub = effort_df[effort_df["student"] == name]
        if len(sub) and sub["frames_reviewed"].sum() > 0:
            rate = sub["total_corrections"] / sub["frames_reviewed"].clip(lower=1)
            ax.plot(sub["round"], rate, "o-", color=student_colors[name], label=name)
    ax.set_xlabel("Round")
    ax.set_ylabel("Corrections per frame")
    ax.set_title("Correction Rate (lower = better model)")
    ax.legend()
    ax.axhline(y=0.25, color="gray", linestyle="--", alpha=0.5, label="Target")

    # FP and FN rates separately
    ax = axes[1]
    for name in students:
        sub = effort_df[effort_df["student"] == name]
        if len(sub) and sub["frames_reviewed"].sum() > 0:
            fp_rate = sub["false_positives"] / sub["frames_reviewed"].clip(lower=1)
            fn_rate = sub["missed_worms"] / sub["frames_reviewed"].clip(lower=1)
            ax.plot(sub["round"], fp_rate, "o--", color=student_colors[name], alpha=0.6)
            ax.plot(sub["round"], fn_rate, "s-", color=student_colors[name],
                    label=f"{name}")
    ax.set_xlabel("Round")
    ax.set_ylabel("Rate per frame")
    ax.set_title("FP (dashed) and FN (solid) Rates")
    ax.legend()

    plt.tight_layout()
    plt.show()


---
## 4. Inter-Annotator Agreement

For months where multiple students overlap (secondary assignments), compare their
final counts. High agreement means the detection pipeline is robust to annotator
variation.


In [ ]:
# ── Find overlapping months ──
# The assignment scheme has overlap: each student's secondary month is
# another student's primary month.

# Collect final counts by month
month_counts = defaultdict(dict)
for name, info in student_data.items():
    if "final_counts" in info:
        month_key = info["month"]
        month_counts[month_key][name] = info["final_counts"]

# Also check secondary month outputs
for name, info in student_data.items():
    for rd in info.get("rounds", []):
        # Check if this round was for a different month (secondary)
        if "counts" in rd:
            pass  # We'd need month metadata per round

overlap_months = {m: s for m, s in month_counts.items() if len(s) >= 2}
if overlap_months:
    print(f"Found {len(overlap_months)} months with multiple annotators:")
    for m, students_dict in overlap_months.items():
        print(f"  {m}: {list(students_dict.keys())}")
else:
    print("No overlapping months found in final exports.")
    print("Inter-annotator agreement requires secondary month assignments.")


In [ ]:
# ── Compare counts for overlapping months ──
if overlap_months:
    for month, students_dict in overlap_months.items():
        names = list(students_dict.keys())
        if len(names) < 2:
            continue

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        fig.suptitle(f"Inter-Annotator Agreement — {month}", fontsize=13)

        # Scatter: student A vs student B counts
        ax = axes[0]
        df_a = students_dict[names[0]].set_index("date")["scaleworm_count"]
        df_b = students_dict[names[1]].set_index("date")["scaleworm_count"]
        merged = pd.DataFrame({"a": df_a, "b": df_b}).dropna()
        if len(merged):
            ax.scatter(merged["a"], merged["b"], alpha=0.5, s=20)
            lim = max(merged["a"].max(), merged["b"].max()) * 1.1
            ax.plot([0, lim], [0, lim], "k--", alpha=0.3)
            ax.set_xlabel(f"{names[0]} count")
            ax.set_ylabel(f"{names[1]} count")
            # Correlation
            r = merged["a"].corr(merged["b"])
            mae = (merged["a"] - merged["b"]).abs().mean()
            ax.set_title(f"r = {r:.3f}, MAE = {mae:.1f}")

        # Time series overlay
        ax = axes[1]
        for n in names:
            df = students_dict[n]
            if "date" in df.columns:
                ax.plot(pd.to_datetime(df["date"]), df["scaleworm_count"],
                        label=n, alpha=0.7)
        ax.set_ylabel("Scale worm count")
        ax.set_title("Daily Time Series")
        ax.legend()
        ax.tick_params(axis="x", rotation=30)

        plt.tight_layout()
        plt.show()
else:
    print("Skipping — no overlapping months to compare.")


---
## 5. Final Count Comparison

Compare baseline (pretrained model) vs student-corrected counts across all months.


In [ ]:
# ── Load baseline counts ──
baseline_path = Path.home() / "scaleworm_starter" / "baseline" / "scaleworm_counts.parquet"
if baseline_path.exists():
    baseline = pd.read_parquet(baseline_path)
    baseline["date"] = pd.to_datetime(baseline["date"])
    print(f"Baseline: {len(baseline)} days, {baseline['scaleworm_count'].sum():.0f} total worms")
else:
    baseline = None
    print(f"Baseline not found at {baseline_path}")


In [ ]:
# ── Combined time series: baseline + all students ──
fig, ax = plt.subplots(figsize=(14, 5))

if baseline is not None:
    ax.fill_between(baseline["date"], baseline["scaleworm_count"],
                    alpha=0.15, color="gray", label="2023 baseline")

for name, info in student_data.items():
    if "final_counts" in info:
        df = info["final_counts"]
        if "date" in df.columns:
            dates = pd.to_datetime(df["date"])
            ax.plot(dates, df["scaleworm_count"], "o-", markersize=3,
                    label=f"{name} ({info['month']})")

ax.set_ylabel("Scale worm count")
ax.set_title("Population Time Series — All Students")
ax.legend()
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


---
## 6. Report Cards

Summary statistics for each student's work.


In [ ]:
# ── Summary table ──
summary_rows = []
for name, info in student_data.items():
    row = {"student": name, "month": info["month"]}

    # Total rounds
    row["rounds"] = len(info["rounds"])

    # Total corrections
    total_fp = sum(r["n_fps"] for r in info["rounds"])
    total_fn = sum(r["n_fns"] for r in info["rounds"])
    total_frames = sum(r["n_frames_reviewed"] for r in info["rounds"])
    row["frames_reviewed"] = total_frames
    row["total_fp"] = total_fp
    row["total_fn"] = total_fn
    row["total_corrections"] = total_fp + total_fn

    # Convergence
    if info.get("report_card"):
        rc = info["report_card"]
        row["converged"] = rc.get("converged", False)
        row["final_mae"] = rc.get("mae", None)
    else:
        row["converged"] = None
        row["final_mae"] = None

    # Has final
    row["has_final"] = info["has_final"]

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))


In [ ]:
# ── Quality ranking ──
if len(summary):
    # Score: more corrections + convergence = better
    summary["quality_score"] = (
        summary["total_corrections"].rank(ascending=False) * 0.3 +
        summary["frames_reviewed"].rank(ascending=False) * 0.3 +
        summary["rounds"].rank(ascending=False) * 0.2 +
        summary["converged"].fillna(False).astype(int).rank(ascending=False) * 0.2
    )
    summary_ranked = summary.sort_values("quality_score")
    print("Student ranking (lower score = better):")
    print(summary_ranked[["student", "month", "total_corrections", "rounds",
                           "converged", "quality_score"]].to_string(index=False))


---
## 7. Cumulative Training Effect

Since later students can start with models improved by earlier students, we expect
decreasing initial error rates across the student sequence. This measures the value
of each student's contribution to the shared model.


In [ ]:
# ── Cumulative training effect ──
# Compare round-1 correction rates for each student (in assignment order)
student_order = list(STUDENTS.keys())  # in assignment order
round1_rates = []

for name in student_order:
    if name not in student_data:
        continue
    info = student_data[name]
    r1 = [r for r in info["rounds"] if r["round"] == 1]
    if r1 and r1[0]["n_frames_reviewed"] > 0:
        rate = (r1[0]["n_fps"] + r1[0]["n_fns"]) / r1[0]["n_frames_reviewed"]
        round1_rates.append({"student": name, "round1_correction_rate": rate})

if round1_rates:
    fig, ax = plt.subplots(figsize=(8, 4))
    names = [r["student"] for r in round1_rates]
    rates = [r["round1_correction_rate"] for r in round1_rates]
    ax.bar(range(len(names)), rates, color=plt.cm.tab10(np.arange(len(names))))
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_ylabel("Corrections per frame (round 1)")
    ax.set_title("Cumulative Training Effect\n(lower = previous students improved the model)")
    plt.tight_layout()
    plt.show()

    if len(rates) >= 2:
        improvement = (rates[0] - rates[-1]) / rates[0] * 100
        print(f"\nFirst student round-1 rate: {rates[0]:.2f}")
        print(f"Last student round-1 rate:  {rates[-1]:.2f}")
        print(f"Improvement: {improvement:.0f}%")
else:
    print("Not enough round-1 data to assess cumulative training effect.")


---
## 8. Export Summary

Save cross-student comparison data for the manuscript.


In [ ]:
# ── Export comparison data ──
output_dir = Path.home() / "scaleworm_students" / "comparison"
output_dir.mkdir(parents=True, exist_ok=True)

if len(effort_df):
    effort_df.to_csv(output_dir / "labeling_effort.csv", index=False)
    print(f"Saved: {output_dir / 'labeling_effort.csv'}")

if len(summary):
    summary.to_csv(output_dir / "student_summary.csv", index=False)
    print(f"Saved: {output_dir / 'student_summary.csv'}")

print(f"\nComparison outputs in: {output_dir}")
